<a href="https://colab.research.google.com/github/Kkkk098/anova/blob/main/anova_R.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Загружаем данные.

In [ ]:
url <- "https://raw.githubusercontent.com/Kkkk098/anova/refs/heads/main/data/survey_results.csv"
df <- read.csv(url)

В датасете переменная стиль чат-бота является категориальной, уровень экстраверсии - дискретная величина от 0 до 24, уровень антропоморфизма - непрерывная величина от 1 до 5 (непрерывная, потому что во-первых измеряли шкалой VAS, а во-вторых, так как это среднее ответов человека), уровень эмоционального отношения - непрерывная величина от 1 до 5 по тем же причинам.

In [ ]:
head(df)

,participant_id,style,age,gender,chatbot_experience,extraversion,anthropomorphism,emotional_attitude
,<int>,<chr>,<int>,<chr>,<chr>,<int>,<dbl>,<dbl>
1,1,neutral,24,male,high,9,2.05,1.95
2,2,neutral,23,male,medium,11,2.40,2.30
3,3,neutral,23,male,low,10,2.25,2.15
4,4,neutral,24,male,high,13,2.75,2.65
5,5,neutral,24,female,medium,14,3.10,2.95
6,6,neutral,23,male,low,16,2.95,2.85


Переменная стиль чат-бота является категориальной, при этом ее можно рассматривать как порядковую, так как от наименьшего (нейтральный стиль) к наибольшему (эмоционально-антропоморфный стиль).

In [ ]:
df$style_ord <- ordered(df$style, levels = c("neutral", "moderate", "emotional"))

---
**Сначала посмотрим влияние стиля чат-бота и уровня экстраверсии на уровень антропоморфности (в нашей аннотации это гипотеза 1 и часть третьей).**

Проведем двухфакторный дисперсионный анализ, проверяя гипотезу за гипотезой.

In [ ]:
q <- lm(anthropomorphism ~ style_ord * extraversion, data = df)    # с учетом всех взаимодействий
q12 <- lm(anthropomorphism ~ style_ord + extraversion, data = df)  # аддитивная, отсутствует А*В
q1 <- lm(anthropomorphism ~ extraversion, data = df)                # отсутствует А
q2 <- lm(anthropomorphism ~ style_ord, data = df)                # отсутствует B
q0 <- lm(anthropomorphism ~ 1, data = df)                           # отсутствует все

In [ ]:
anova(q12,q)


,Res.Df,RSS,Df,Sum of Sq,F,Pr(>F)
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,14,0.3657923,NA,NA,NA,NA
2,12,0.2485723,2,0.1172199,2.829436,0.09847129


Так как принимаем аддитивную модель, в следующем анализе именно аддитивную модель можем взять за полную и сравнивать с ней остальные модели.

In [ ]:
anova(q1,q12)
print("-------------------------------------------------------------------------")
anova(q2,q12)
print("-------------------------------------------------------------------------")
anova(q0,q12)

,Res.Df,RSS,Df,Sum of Sq,F,Pr(>F)
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,16,8.2058879,NA,NA,NA,NA
2,14,0.3657923,2,7.840096,150.0323,3.497538e-10


[1] "-------------------------------------------------------------------------"


,Res.Df,RSS,Df,Sum of Sq,F,Pr(>F)
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,15,1.8854167,NA,NA,NA,NA
2,14,0.3657923,1,1.519624,58.16072,2.379144e-06


[1] "-------------------------------------------------------------------------"


,Res.Df,RSS,Df,Sum of Sq,F,Pr(>F)
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,17,10.0540278,NA,NA,NA,NA
2,14,0.3657923,3,9.688236,123.5996,2.608876e-10


Нормальное оформление таблицы для вставки в нашу работу.

In [ ]:
res1 <- anova(q12, q)
res2 <- anova(q1, q12)
res3 <- anova(q2, q12)
res4 <- anova(q0, q12)

extract <- function(res, name) {
  data.frame(
    NullHypothesis = name,
    Df = res$Df[2],
    SumSq = res$`Sum of Sq`[2],
    F = res$F[2],
    p = res$`Pr(>F)`[2]
  )
}

table_all <- rbind(
  extract(res1, "anthropomorphism ~ style + extraversion"),
  extract(res2, "anthropomorphism ~ extraversion"),
  extract(res3, "anthropomorphism ~ style"),
  extract(res4, "anthropomorphism ~ 1")
)

format_p <- function(p) {
  if (p < 0.001) {
    return("<.001")
  } else {
    return(sprintf("%.3f", p))
  }
}

table_all$p <- sapply(table_all$p, format_p)
table_all$SumSq <- round(table_all$SumSq, 2)
table_all$F <- round(table_all$F, 2)

table_all

NullHypothesis,Df,SumSq,F,p
<chr>,<dbl>,<dbl>,<dbl>,<chr>
anthropomorphism ~ style + extraversion,2,0.12,2.83,0.098
anthropomorphism ~ extraversion,2,7.84,150.03,<.001
anthropomorphism ~ style,1,1.52,58.16,<.001
anthropomorphism ~ 1,3,9.69,123.60,<.001


In [ ]:
library(sjPlot)

# Создаем красивую таблицу для ANOVA (появится файл для скачивания)
tab_df(table_all,
       title = "Результаты проверки гипотез двухфакторного дисперсионного анализа влияния стиля общения и экстраверсии на антропоморфизм",
       file = "anova_table.html",
       digits = 2,
       show.rownames = FALSE)

Можем также по критерию Акайке и Баесовскому посмотреть наиболее подходящую модель (это больше для проверки, в отчет предлагаю не вставлять).

In [ ]:
aic <- AIC(q12, q1, q2, q0, q)$AIC
bic <- BIC(q12, q1, q2, q0, q)$BIC

In [ ]:
aic

bic


[1]  -9.047319  42.942430  18.469775  44.598615 -12.001290

[1] -4.595461 45.613546 22.031262 46.379359 -5.768688

Вывод следующий, на основе p-value все гипотезу отвергаем, кроме первой, другими словами, оба фактора - стиль чат-бота и уровень экстравертности, влияют на уровень антропоморфизма, но независимо друг от друга (модель аддитивная). Самые низкие значения критериев, однако, указывают на полную модель, то есть с взаимодействием... Игнорирую этот факт.

Раз оба фактора влияют, можно посмотреть, какой из них оказывает бОльшее влияние на целевую переменную.

Для этого воспользуемся регрессионным анализом, но для того, чтобы сравнивать силу влияния обеих переменных - сначала нам нужно их стандантизировать.

In [ ]:
df$anthro_z <- scale(df$anthropomorphism)
df$extra_z <- scale(df$extraversion)
df$style_num <- as.numeric(df$style_ord)
df$style_z <- scale(df$style_num)

model_z <- lm(anthro_z ~ style_z + extra_z, data = df)
summary(model_z)


Call:
lm(formula = anthro_z ~ style_z + extra_z, data = df)

Residuals:
     Min       1Q   Median       3Q      Max 
-0.29305 -0.15194 -0.00518  0.10495  0.45041 

Coefficients:
             Estimate Std. Error t value Pr(>|t|)    
(Intercept) 5.839e-17  4.792e-02   0.000        1    
style_z     8.839e-01  4.936e-02  17.907 1.56e-11 ***
extra_z     3.892e-01  4.936e-02   7.884 1.03e-06 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Residual standard error: 0.2033 on 15 degrees of freedom
Multiple R-squared:  0.9635,	Adjusted R-squared:  0.9587 
F-statistic: 198.1 on 2 and 15 DF,  p-value: 1.64e-11


Смотрим на коэффициенты Estimate, так как у переменной стиль коэффициент больше, следовательно, он в бОльшей степени влияет на антропоморфизм.

Учитывая, что коэффициенты положительные, делаем вывод, что увеличение уровня стилизации бота увеличивает уровень антропоморфизма и увеличение уровня экстраверсии человека также увеличивает значение антропоморфизма.

In [ ]:
# Установить, если не установлен
install.packages("sjPlot")



Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘rbibutils’, ‘Rdpack’, ‘reformulas’, ‘effectsize’, ‘bayestestR’, ‘datawizard’, ‘ggeffects’, ‘insight’, ‘parameters’, ‘performance’, ‘sjlabelled’, ‘sjmisc’, ‘sjstats’




In [ ]:
library(sjPlot)
# Вывод красиво оформленной таблицы регрессии
tab_model(model_z, show.se = TRUE, show.ci = FALSE,
          show.p = TRUE, show.std = TRUE,
          title = "Регрессия стандартизированных переменных",
          dv.labels = "Anthropomorphism (z)",
          file = "regression_table.html")


---
**Перейдем к изучению влияния стиля и уровня экстравертности на уровень эмоционального отношения к чат-боту (в нашей работе это гипотезы 2 и вторая часть 3ей).**



In [ ]:
q <- lm(emotional_attitude ~ style_ord * extraversion, data = df)    # с учетом всех взаимодействий
q12 <- lm(emotional_attitude ~ style_ord + extraversion,data = df)  # аддитивная, отсутствует А*В
q1 <- lm(emotional_attitude ~ extraversion, data = df)                # отсутствует А
q2 <- lm(emotional_attitude ~ style_ord, data = df)                # отсутствует B
q0 <- lm(emotional_attitude ~ 1, data = df)                           # отсутствует все

In [ ]:
anova(q12,q)


,Res.Df,RSS,Df,Sum of Sq,F,Pr(>F)
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,14,0.2921227,NA,NA,NA,NA
2,12,0.1996779,2,0.09244488,2.777821,0.101997


Так как принимаем аддитивную модель, в следующем анализе именно аддитивную модель можем взять за полную и сравнивать с ней остальные модели.

In [ ]:
anova(q1,q12)
print("-------------------------------------------------------------------------")
anova(q2,q12)
print("-------------------------------------------------------------------------")
anova(q0,q12)

,Res.Df,RSS,Df,Sum of Sq,F,Pr(>F)
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,16,9.4858144,NA,NA,NA,NA
2,14,0.2921227,2,9.193692,220.3041,2.626849e-11


[1] "-------------------------------------------------------------------------"


,Res.Df,RSS,Df,Sum of Sq,F,Pr(>F)
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,15,1.8991667,NA,NA,NA,NA
2,14,0.2921227,1,1.607044,77.01768,4.587683e-07


[1] "-------------------------------------------------------------------------"


,Res.Df,RSS,Df,Sum of Sq,F,Pr(>F)
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,17,11.4444444,NA,NA,NA,NA
2,14,0.2921227,3,11.15232,178.1586,2.193349e-11


In [ ]:
res1 <- anova(q12, q)
res2 <- anova(q1, q12)
res3 <- anova(q2, q12)
res4 <- anova(q0, q12)

extract <- function(res, name) {
  data.frame(
    NullHypothesis = name,
    Df = res$Df[2],
    SumSq = res$`Sum of Sq`[2],
    F = res$F[2],
    p = res$`Pr(>F)`[2]
  )
}

table_all <- rbind(
  extract(res1, "emotional_attitude ~ style + extraversion"),
  extract(res2, "emotional_attitude ~ extraversion"),
  extract(res3, "emotional_attitude ~ style"),
  extract(res4, "emotional_attitude ~ 1")
)

format_p <- function(p) {
  if (p < 0.001) {
    return("<.001")
  } else {
    return(sprintf("%.3f", p))
  }
}

table_all$p <- sapply(table_all$p, format_p)
table_all$SumSq <- round(table_all$SumSq, 2)
table_all$F <- round(table_all$F, 2)

table_all

NullHypothesis,Df,SumSq,F,p
<chr>,<dbl>,<dbl>,<dbl>,<chr>
emotional_attitude ~ style + extraversion,2,0.09,2.78,0.102
emotional_attitude ~ extraversion,2,9.19,220.30,<.001
emotional_attitude ~ style,1,1.61,77.02,<.001
emotional_attitude ~ 1,3,11.15,178.16,<.001


In [ ]:
tab_df(table_all,
       title = "Результаты проверки гипотез двухфакторного дисперсионного анализа влияния стиля общения и экстраверсии на эмоциональное отношение",
       file = "anova_table_2.html",
       digits = 2,
       show.rownames = FALSE)

In [ ]:
aic <- AIC(q12, q1, q2, q0, q)$AIC
aic
bic <- BIC(q12, q1, q2, q0, q)$BIC
bic

[1] -13.09537  45.55145  18.60057  46.93017 -15.94380

[1] -8.643508 48.222565 22.162056 48.710918 -9.711201

In [ ]:
df$emotional_z <- scale(df$emotional_attitude)

model_z <- lm(emotional_z ~ style_z + extra_z, data = df)
summary(model_z)


Call:
lm(formula = emotional_z ~ style_z + extra_z, data = df)

Residuals:
     Min       1Q   Median       3Q      Max 
-0.30691 -0.11020  0.02162  0.06573  0.38685 

Coefficients:
              Estimate Std. Error t value Pr(>|t|)    
(Intercept) -1.150e-16  4.127e-02   0.000        1    
style_z      8.963e-01  4.251e-02  21.086 1.46e-12 ***
extra_z      3.736e-01  4.251e-02   8.788 2.66e-07 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Residual standard error: 0.1751 on 15 degrees of freedom
Multiple R-squared:  0.9729,	Adjusted R-squared:  0.9693 
F-statistic: 269.8 on 2 and 15 DF,  p-value: 1.743e-12


In [ ]:
# Вывод красиво оформленной таблицы регрессии
tab_model(model_z, show.se = TRUE, show.ci = FALSE,
          show.p = TRUE, show.std = TRUE,
          title = "Регрессия стандартизированных переменных",
          dv.labels = "Emotional attitude (z)",
          file = "regression_table_2.html")